cabins higher up imply class, maybe fill nulls by an estimate based on classes

In [1]:
import numpy as np
from xgboost import XGBClassifier
import pandas as pd
from category_encoders import MEstimateEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import KFold, cross_val_score
from xgboost import XGBRegressor

o_encoder = OrdinalEncoder()

In [2]:
def make_mi_scores(X, y):
    X = X.copy()
    for colname in X.select_dtypes(["object", "category"]):
        X[colname], _ = X[colname].factorize()
    discrete_features = [pd.api.types.is_integer_dtype(t) for t in X.dtypes]
    mi_scores = mutual_info_classif(X, y, discrete_features=discrete_features, random_state=0)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores

In [3]:
df_train = pd.read_csv('train.csv', index_col='PassengerId')
df_test = pd.read_csv('test.csv')

In [4]:
X_train = df_train.drop('Survived', axis='columns')
y_train = df_train['Survived']

X_test = df_test

# 1. Fill missing values first with the most common port ('S')
X_train['Embarked'] = X_train['Embarked'].fillna('S')
X_test['Embarked'] = X_test['Embarked'].fillna('S')

# 2. Apply One-Hot Encoding
X_train = pd.get_dummies(X_train, columns=['Embarked'], prefix='Embarked')
X_test = pd.get_dummies(X_test, columns=['Embarked'], prefix='Embarked')


In [5]:
# 1. Combine train and test tickets to get true group sizes for everyone
all_tickets = pd.concat([df_train['Ticket'], df_test['Ticket']])
ticket_counts = all_tickets.value_counts()

# 2. Map the comprehensive counts to both feature sets
X_train['TicketGroupSize'] = df_train['Ticket'].map(ticket_counts)
X_test['TicketGroupSize'] = df_test['Ticket'].map(ticket_counts)

# 3. Create FarePerPerson by dividing the total Fare by the TicketGroupSize
X_train['FarePerPerson'] = df_train['Fare'] / X_train['TicketGroupSize']
X_test['FarePerPerson'] = df_test['Fare'] / X_test['TicketGroupSize']

# 4. Drop the original Ticket and Fare columns (since we are using FarePerPerson now)
X_train = X_train.drop(['Ticket', 'Fare'], axis='columns', errors='ignore')
X_test = X_test.drop(['Ticket', 'Fare'], axis='columns', errors='ignore')

In [6]:
# Family/group survival rate: passengers sharing a ticket often traveled as a family or
# group and tended to survive or die together. Computed OUT-OF-FOLD for X_train, using the
# same cv splitter as model selection below, so a row's feature never depends on the target
# of another row in the same held-out fold (that was leaking across CV folds before).
# For X_test we can safely fit on the full training set, since test labels are never used.
from sklearn.model_selection import StratifiedKFold

# Shared CV splitter -- defined here (rather than down where it's used) so this
# out-of-fold encoding and the later model selection see identical fold boundaries.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def family_survival_rate(tickets, group_sum, group_count, global_rate):
    total_count = tickets.map(group_count).fillna(0.0)
    total_sum = tickets.map(group_sum).fillna(0.0)
    known = total_count > 0
    safe_count = total_count.where(known, 1)
    rate = (total_sum / safe_count).where(known, global_rate)
    return rate, known.astype(int)

X_train['FamilySurvivalRate'] = np.nan
X_train['FamilySurvivalKnown'] = 0

for train_idx, hold_idx in cv.split(X_train, y_train):
    # Group stats built ONLY from this fold's training rows
    fold_tickets = df_train['Ticket'].iloc[train_idx]
    fold_y = y_train.iloc[train_idx]
    fold_group_sum = fold_y.groupby(fold_tickets).sum()
    fold_group_count = fold_tickets.value_counts()
    fold_global_rate = fold_y.mean()

    hold_tickets = df_train['Ticket'].iloc[hold_idx]
    rate, known = family_survival_rate(hold_tickets, fold_group_sum, fold_group_count, fold_global_rate)
    X_train.iloc[hold_idx, X_train.columns.get_loc('FamilySurvivalRate')] = rate.values
    X_train.iloc[hold_idx, X_train.columns.get_loc('FamilySurvivalKnown')] = known.values

# Test set: fit on the FULL training set -- no leakage risk since test labels are never used
group_sum = y_train.groupby(df_train['Ticket']).sum()
group_count = df_train['Ticket'].value_counts()
global_survival_rate = y_train.mean()
X_test['FamilySurvivalRate'], X_test['FamilySurvivalKnown'] = family_survival_rate(
    df_test['Ticket'], group_sum, group_count, global_survival_rate
)

In [7]:
X_train['Cabin'] = X_train.Cabin.fillna('U')
X_test['Cabin'] = X_test.Cabin.fillna('U')
X_train['Cabin'] = X_train['Cabin'].str[0]
X_test['Cabin'] = X_test['Cabin'].str[0]

# Convert the text in Cabin to numerical categories. Fit the encoder on train only and
# reuse it on test, so the same deck letter always maps to the same code in both splits.
cabin_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train['Cabin'] = cabin_encoder.fit_transform(X_train[['Cabin']])
X_test['Cabin'] = cabin_encoder.transform(X_test[['Cabin']])

X_train['Sex'] = o_encoder.fit_transform(X_train[['Sex']])
X_test['Sex'] = o_encoder.transform(X_test[['Sex']])

# Calculate total family size
X_train['FamilySize'] = X_train['SibSp'] + X_train['Parch'] + 1
X_test['FamilySize'] = X_test['SibSp'] + X_test['Parch'] + 1

# Create a binary column for passengers who are traveling alone
X_train['IsAlone'] = 0
X_test['IsAlone'] = 0
X_train.loc[X_train['FamilySize'] == 1, 'IsAlone'] = 1
X_test.loc[X_test['FamilySize'] == 1, 'IsAlone'] = 1

In [8]:
# 1. Select the continuous/numerical features to cluster on
cluster_features = ['Age', 'FarePerPerson', 'Pclass', 'FamilySize']

# 2. Scale the features so they have a mean of 0 and variance of 1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[cluster_features].fillna(0))
X_test_scaled = scaler.transform(X_test[cluster_features].fillna(0))

# 3. Fit K-Means (let's try grouping passengers into 4 distinct archetypes)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

# 4. Assign the cluster labels as a new feature to both datasets
X_train['PassengerCluster'] = kmeans.fit_predict(X_train_scaled)
X_test['PassengerCluster'] = kmeans.predict(X_test_scaled)

# Optional: Convert to categorical or leave as integer so XGBoost can split on it
X_train['PassengerCluster'] = X_train['PassengerCluster'].astype('category')
X_test['PassengerCluster'] = X_test['PassengerCluster'].astype('category')

In [9]:
X_train['Title'] = (
    X_train['Name']
    .str.split(',')
    .str[1]
    .str.split('.')
    .str[0]
    .str.strip()
)
X_test['Title'] = (
    X_test['Name']
    .str.split(',')
    .str[1]
    .str.split('.')
    .str[0]
    .str.strip()
)
X_train = X_train.drop('Name', axis='columns')
X_test = X_test.drop('Name', axis='columns')
X_train['Title'], i = X_train['Title'].factorize()
X_test['Title'], _ = X_test['Title'].factorize()

In [10]:
# Calculate the median age for each title and fill missing values
X_train['Age'] = X_train['Age'].fillna(
    X_train.groupby('Title')['Age'].transform('median')
)
X_test['Age'] = X_test['Age'].fillna(
    X_test.groupby('Title')['Age'].transform('median')
)

# Fallback just in case a rare title has absolutely no known ages in the dataset
X_train['Age'] = X_train['Age'].fillna(X_train['Age'].median())
X_test['Age'] = X_test['Age'].fillna(X_test['Age'].median())

In [11]:
print(make_mi_scores(X_train, y_train))

Title                  0.178046
Sex                    0.141809
FarePerPerson          0.123849
PassengerCluster       0.058754
TicketGroupSize        0.058605
Pclass                 0.058107
Cabin                  0.055185
FamilySize             0.047781
FamilySurvivalRate     0.043788
Age                    0.035728
SibSp                  0.023197
Embarked_C             0.021928
IsAlone                0.020593
Embarked_S             0.020229
Parch                  0.016366
FamilySurvivalKnown    0.015114
Embarked_Q             0.000000
Name: MI Scores, dtype: float64


In [12]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# 1. Separate target and features for training
X = X_train.copy()
y = y_train.copy()

# Note: `cv` was already defined above (family-survival-rate cell) and is reused here so
# every step, including that feature's out-of-fold encoding, shares the same fold boundaries.

# 3. Tune XGBoost
xgb_param_dist = {
    'n_estimators': [100, 150, 200, 300, 400],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1, 5],
    'reg_lambda': [0.5, 1, 2, 5, 10],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss', tree_method='hist'),
    param_distributions=xgb_param_dist,
    n_iter=60,
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
)
xgb_search.fit(X, y)
best_xgb = xgb_search.best_estimator_

print(f"Best XGBoost CV Accuracy: {xgb_search.best_score_:.4f}")
print(f"Best XGBoost params: {xgb_search.best_params_}")

Best XGBoost CV Accuracy: 0.8541
Best XGBoost params: {'subsample': 0.8, 'reg_lambda': 5, 'reg_alpha': 0.5, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.6}


In [13]:
from sklearn.ensemble import RandomForestClassifier

# Tune Random Forest (kept alongside XGBoost specifically to compare which performs better)
rf_param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3, 4, 5, 6, 8, None],
    'min_samples_split': [2, 4, 6, 8, 10],
    'min_samples_leaf': [1, 2, 3, 5, 8],
    'max_features': ['sqrt', 'log2', None],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=rf_param_dist,
    n_iter=60,
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
)
rf_search.fit(X, y)
best_rf = rf_search.best_estimator_

print(f"Best Random Forest CV Accuracy: {rf_search.best_score_:.4f}")
print(f"Best Random Forest params: {rf_search.best_params_}")

Best Random Forest CV Accuracy: 0.8541
Best Random Forest params: {'n_estimators': 100, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 5}


In [14]:
from sklearn.ensemble import VotingClassifier

ensemble_model = VotingClassifier(
    estimators=[('xgb', best_xgb), ('rf', best_rf)],
    voting='soft',
)

# Score every candidate with the SAME cv splitter so per-fold scores are directly
# comparable. On a dataset this small, the worst fold tends to track leaderboard
# accuracy better than the mean, since a high mean can hide a model that got lucky
# on one split.
xgb_fold_scores = cross_val_score(best_xgb, X, y, cv=cv, scoring='accuracy')
rf_fold_scores = cross_val_score(best_rf, X, y, cv=cv, scoring='accuracy')
ensemble_fold_scores = cross_val_score(ensemble_model, X, y, cv=cv, scoring='accuracy')

candidates = {
    'xgb': (xgb_fold_scores, best_xgb),
    'rf': (rf_fold_scores, best_rf),
    'ensemble': (ensemble_fold_scores, ensemble_model),
}

print("Model comparison (5-fold CV accuracy):")
for name, (scores, _) in candidates.items():
    print(f"  {name:9s} mean={scores.mean():.4f}  min={scores.min():.4f}  max={scores.max():.4f}  std={scores.std():.4f}  folds={np.round(scores, 4)}")

Model comparison (5-fold CV accuracy):
  xgb       mean=0.8541  min=0.8371  max=0.8603  std=0.0088  folds=[0.8603 0.8596 0.8371 0.8539 0.8596]
  rf        mean=0.8541  min=0.8315  max=0.8659  std=0.0119  folds=[0.8659 0.8539 0.8315 0.8596 0.8596]
  ensemble  mean=0.8541  min=0.8315  max=0.8652  std=0.0119  folds=[0.8603 0.8652 0.8315 0.8596 0.8539]


In [15]:
# Select whichever candidate has the highest MINIMUM fold score (best worst-case),
# rather than the highest mean -- empirically this has tracked leaderboard accuracy
# better than mean CV accuracy for this dataset.
best_name, (best_scores, best_model) = max(candidates.items(), key=lambda item: item[1][0].min())
print(f"Selected '{best_name}' for submission (min fold accuracy {best_scores.min():.4f}, mean {best_scores.mean():.4f})")

best_model.fit(X, y)
test_preds = best_model.predict(X_test.drop('PassengerId', axis='columns'))

submission = pd.DataFrame({
    'PassengerId': df_test['PassengerId'],
    'Survived': test_preds
})
submission.to_csv('submission.csv', index=False)
print("Submission saved successfully!")

Selected 'xgb' for submission (min fold accuracy 0.8371, mean 0.8541)
Submission saved successfully!
